In [27]:
%reset -f

In [28]:
import os
import pandas as pd
import xlwings as xw
import numpy as np

In [ ]:
batch_key_file = r"/Users/morgangodley/Documents/Projects Folder/Dummy Data/BatchKey.xlsx"
key_df = pd.read_excel(batch_key_file, sheet_name="Sheet1", engine="openpyxl")

base_path = r"/Users/morgangodley/Documents/Projects Folder/Dummy Data"

In [37]:
for i, row in key_df.iterrows():
    client = row["client"]
    client_ISL = row["client_ISL"]
    threshold = row["threshold"]
    tab_experience = row["tab_experience"]
    tab_claims = row["tab_claims"]
    header_claims = row["header_claims"]

    print(f"\nRunning {client}...")

    client_files = [f for f in os.listdir(base_path) if client.lower() in f.lower() and f.endswith(".xls")]
    if not client_files:
        print(f"No .xls files found for client: {client}")
        continue
    else:
        print(f"Found file(s) for {client}: {client_files}")

        # Manipulate the First File found; use xlwings (xw) package to manipulate .xls documents
        client_file1 = client_files[0]
        client_final_xls_path = os.path.join(base_path, client_file1) # Reset path to First File
        client_final_xls = xw.Book(client_final_xls_path)             # Open file with xw package
    
        # Pull Large Claims info
        for tab in client_final_xls.sheets:
            tab_name = tab.name
            if tab_name.startswith(tab_claims):
                df_lc = tab.used_range.options(pd.DataFrame, header=0, index=False).value
                print(f"Large Claims Sheet: {tab_name}, saved as DataFrame df_lc\n")
                #print(df_lc.head(15))

                # Remove Cigna's header
                df_lc_cleaned = df_lc.iloc[header_claims:].reset_index(drop=True)         # Drop first 10 rows
                df_lc_cleaned.columns = df_lc_cleaned.iloc[0]                  # Row 11 becomes headers
                df_lc_cleaned = df_lc_cleaned.iloc[1:].reset_index(drop=True)  # Drop the header row

                # Convert $ columns to numeric
                cols_convert = ["DRUG CLAIMS", "PAID CLAIMS", "CLAIMANT TOTAL"]

                for col in cols_convert:
                    df_lc_cleaned[col] = (
                        df_lc_cleaned[col]
                        .astype(str)                             # convert to string
                        #.str.replace(r"[\$,]", "", regex=True)  # remove $ and commas
                        .replace("None", np.nan)                 # convert 'None' strings to np.nan
                        .astype(float)                           # convert to float
                    )

                # Dominate Diagnosis per Member
                dominant_icd = (df_lc_cleaned.loc
                                # Find the largest CLAIMANT TOTAL for each MEMBER ID
                                [df_lc_cleaned.groupby("MEMBER ID")["CLAIMANT TOTAL"].idxmax()]
                                # Select corresponding ICD DESCRIPTION for max CLAIMANT TOTAL
                                .set_index("MEMBER ID")["ICD DESCRIPTION"])
                # Replace all ICD DESCRIPTION values by MEMBER ID with dominant ICD code
                df_lc_cleaned["ICD DESCRIPTION"] = df_lc_cleaned["MEMBER ID"].map(dominant_icd)
                #print(df_lc_cleaned.head(15))

                # Keep Relevant Columns: MEMBER ID, REL, ICD DESCRIPTION, DRUG CLAIMS, PAID CLAIMS
                df_lc_cleaned = df_lc_cleaned.copy()
                if "MEMBER STATUS" not in df_lc_cleaned.columns:
                    df_lc_cleaned["MEMBER STATUS"] = np.nan  

                df_lc_cleaned_subset = df_lc_cleaned[["MEMBER ID", "MEMBER STATUS", "REL", "ICD DESCRIPTION",
                                                    "DRUG CLAIMS", "PAID CLAIMS", "CLAIMANT TOTAL"
                                                    ]].copy()

                # Drop rows with "MEMBER ID Total" in MEMBER ID column
                df_lc_cleaned_subset = df_lc_cleaned_subset[df_lc_cleaned_subset["MEMBER ID"] != "MEMBER ID Total"]
                df_lc_cleaned_subset = df_lc_cleaned_subset.dropna(subset=["MEMBER ID"]).reset_index(drop=True)

                # Group on MEMBER ID, sum numeric columns, for non-numeric columns, keep the first value
                df_lc_cleaned_subset = df_lc_cleaned_subset.groupby("MEMBER ID").agg(
                    {"MEMBER STATUS": "first",
                    "REL": "first",
                    "ICD DESCRIPTION": "first",
                    "DRUG CLAIMS": "sum",
                    "PAID CLAIMS": "sum",
                    "CLAIMANT TOTAL": "sum"}).reset_index()

                #Re-order columns
                df_lc_cleaned_subset = df_lc_cleaned_subset[
                    ["MEMBER ID", "REL", "MEMBER STATUS", "ICD DESCRIPTION", 
                    "PAID CLAIMS", "DRUG CLAIMS", "CLAIMANT TOTAL"]]

                # Sort CLAIMANT TOTAL in descending order
                df_lc_cleaned_subset = df_lc_cleaned_subset.sort_values("CLAIMANT TOTAL", ascending=False).reset_index(drop=True)
                print(df_lc_cleaned_subset.head(15))

                # Round numeric columns to 2 decimal places
                df_lc_cleaned_subset[["PAID CLAIMS", "DRUG CLAIMS", "CLAIMANT TOTAL"]] = (
                    df_lc_cleaned_subset[["PAID CLAIMS", "DRUG CLAIMS", "CLAIMANT TOTAL"]].round(2))

                # Keep Claims only >= threshold
                df_lc_cleaned_subset = df_lc_cleaned_subset[df_lc_cleaned_subset["CLAIMANT TOTAL"] >= threshold].copy()
                #print(df_lc_cleaned_subset.head(15))

                df_lc_ytd_final = df_lc_cleaned_subset.drop(columns=["MEMBER ID"])

                df_lc_ytd_final

                file_name = f"{client} - Large Claims.xlsx"
                output_path = os.path.join(base_path, file_name)

                # Overwrites file if it already exists
                df_lc_ytd_final.to_excel(output_path, index=False)


        client_final_xls.close()





Running Client_Scale...
Found file(s) for Client_Scale: ['Client_Scale.xls']
Large Claims Sheet: LC-CLIENT_SCALE ASS-1, saved as DataFrame df_lc

0  MEMBER ID REL MEMBER STATUS ICD DESCRIPTION  PAID CLAIMS  DRUG CLAIMS  \
0    11111.0  SB         COBRA     UNSPECIFIED    350001.02      3500.00   
1    33333.0  SP        ACTIVE     AUTOIMMUNE1     95000.00         0.30   
2    44444.0  SB        ACTIVE          ORTHO1     82000.30       900.55   
3    55555.0  SP        ACTIVE     AUTOIMMUNE2     73000.55        60.00   
4    22222.0  CH        ACTIVE     UNSPECIFIED     70000.55        50.00   
5    66666.0  SB         COBRA          KIDNEY     45000.00         0.00   

0  CLAIMANT TOTAL  
0       353501.02  
1        95000.30  
2        82900.85  
3        73060.55  
4        70050.55  
5        45000.00  

Running Client_Tech...
Found file(s) for Client_Tech: ['Client_Tech.xls']
Large Claims Sheet: LC-CLIENT_TECH-1, saved as DataFrame df_lc

0  MEMBER ID REL MEMBER STATUS ICD DESCRI